# Lluvias por departamento en el Perú

In [1]:
%pip install pandas requests lxml


Note: you may need to restart the kernel to use updated packages.


In [2]:
import io
import time
import requests
import pandas as pd

In [3]:
# Importamos la tabla de Wikipedia con los departamentos del Perú y sus capitales.

url = "https://es.wikipedia.org/wiki/Departamentos_del_Per%C3%BA"

respuesta = requests.get(url, headers={"User-Agent": "Mozilla/5.0"})
print(respuesta.status_code)

200


In [4]:
# Importamos la tabla de Wikipedia con los departamentos del Perú y sus capitales.
tablas = pd.read_html(io.StringIO(respuesta.text))

In [5]:
for i, t in enumerate(tablas):
    print(i, t.shape, list(t.columns)[:6])

0 (1, 1) ['Departamentos del Perú (con las provincias de régimen especial)']
1 (24, 11) ['Bandera', 'Escudo', 'Departamento', 'Nombre oficial', 'Abreviatura Nacional', 'Capital']
2 (8, 5) ['Departamento', 'Capital', 'Creación', 'Desaparición', 'Causa']
3 (2, 11) ['Bandera', 'Escudo', 'Provincia', 'Nombre oficial', 'Abreviatura Nacional', 'Capital']
4 (6, 4) ['Provincia', 'Capital', 'Creación', 'Desaparición']
5 (1, 2) [0, 1]


In [6]:
df = tablas[1]
df.head(30)

,Bandera,Escudo,Departamento,Nombre oficial,Abreviatura Nacional,Capital,Ciudad más poblada,Superficie (km²),Población (2025),Creación,Ubicación
0,NaN,NaN,Amazonas,Departamento de Amazonas,AMA,Chachapoyas,Bagua Grande,39 200,467 688,21 de noviembre de 1832 (193 años),NaN
1,NaN,NaN,Áncash,Departamento de Áncash,ANC,Huaraz,Chimbote,35 900,1 205 923,12 de febrero de 1821 (205 años)[nota 1],NaN
2,NaN,NaN,Apurímac,Departamento de Apurímac Apurimaq suyu,APU,Abancay,Andahuaylas,20 900,467 513,28 de abril de 1873 (153 años),NaN
3,NaN,NaN,Arequipa,Departamento de Arequipa Ariqipa suyu,ARE,Arequipa,Arequipa,63 300,1 814 554,26 de abril de 1822 (204 años),NaN
4,NaN,NaN,Ayacucho,Departamento de Ayacucho,AYA,Ayacucho,Ayacucho,43 800,709 026,26 de abril de 1822 (204 años)[nota 2],NaN
5,NaN,NaN,Cajamarca,Departamento de Cajamarca,CAJ,Cajamarca,Cajamarca,33 300,1 490 732,11 de febrero de 1855 (171 años),NaN
6,NaN,NaN,Cusco,Departamento del Cusco,CUS,Cusco,Cusco,72 000,1 379 167,26 de abril de 1822 (204 años),NaN
7,NaN,NaN,Huancavelica,Departamento de Huancavelica,HUV,Huancavelica,Huancavelica,22 100,372 158,26 de abril de 1822 (204 años),NaN
8,NaN,NaN,Huánuco,Departamento de Huánuco,HUC,Huánuco,Huánuco,37 300,807 393,24 de enero de 1869 (157 años),NaN
9,NaN,NaN,Ica,Departamento de Ica,ICA,Ica,Ica,21 300,1 033 084,30 de enero de 1866 (160 años),NaN


In [7]:
deptos = df[["Departamento", "Capital"]].copy()
deptos.columns = ["departamento", "capital"]
deptos

,departamento,capital
0,Amazonas,Chachapoyas
1,Áncash,Huaraz
2,Apurímac,Abancay
3,Arequipa,Arequipa
4,Ayacucho,Ayacucho
5,Cajamarca,Cajamarca
6,Cusco,Cusco
7,Huancavelica,Huancavelica
8,Huánuco,Huánuco
9,Ica,Ica


In [8]:
len(deptos)

24

In [9]:
callao = pd.DataFrame({"departamento": ["Callao"], "capital": ["Callao"]})
deptos = pd.concat([deptos, callao], ignore_index=True)

# El Callao no aparece en la tabla porque es una Provincia Constitucional, no un departamento. Lo agregamos.

len(deptos)

25

In [10]:
deptos[deptos["capital"].str.contains("Huacho")]

# Corregimos también la Capital de Lima. Figura que es Huacho, así que la cambiaremos a Lima.

,departamento,capital
13,Lima,Huacho (de facto)


In [11]:
deptos.loc[deptos["departamento"] == "Lima", "capital"] = "Lima"

# Corroboramos que se haya corregido el error:

print(len(deptos))
deptos

25


,departamento,capital
0,Amazonas,Chachapoyas
1,Áncash,Huaraz
2,Apurímac,Abancay
3,Arequipa,Arequipa
4,Ayacucho,Ayacucho
5,Cajamarca,Cajamarca
6,Cusco,Cusco
7,Huancavelica,Huancavelica
8,Huánuco,Huánuco
9,Ica,Ica


In [12]:
# Ahora, obtendremos la latitud y longitud por cada capital con la API de geocodificación

url_geo = "https://geocoding-api.open-meteo.com/v1/search"

parametros = {"name": "Chachapoyas", "count": 10, "language": "es"}
respuesta = requests.get(url_geo, params=parametros)
datos = respuesta.json()

datos

{'results': [{'id': 3698540,
   'name': 'Chachapoyas',
   'latitude': -6.23169,
   'longitude': -77.86903,
   'elevation': 2360.0,
   'feature_code': 'PPLA',
   'country_code': 'PE',
   'admin1_id': 3699699,
   'admin2_id': 3698538,
   'admin3_id': 8350561,
   'timezone': 'America/Lima',
   'population': 32026,
   'country_id': 3932488,
   'country': 'Perú',
   'admin1': 'Departamento de Amazonas',
   'admin2': 'Provincia de Chachapoyas',
   'admin3': 'Chachapoyas'},
  {'id': 3859420,
   'name': 'Destilería Chachapoyas',
   'latitude': -24.74735,
   'longitude': -65.38527,
   'elevation': 1252.0,
   'feature_code': 'PPL',
   'country_code': 'AR',
   'admin1_id': 3838231,
   'admin2_id': 3862749,
   'timezone': 'America/Argentina/Salta',
   'country_id': 3865483,
   'country': 'Argentina',
   'admin1': 'Provincia de Salta',
   'admin2': 'Departamento Capital'},
  {'id': 13255311,
   'name': 'Chachapoyas',
   'latitude': -8.97357,
   'longitude': -78.49754,
   'elevation': 156.0,
   'fea

In [13]:
# Vemos que la API ha incluído localidades de otros países. Por ejemplo, incluyó un "Chachapoyas" en Argentina y otro en Áncash.
for r in datos["results"]:
    print(r["name"], "|", r["country_code"], "|", r.get("admin1"), "|", r["latitude"], r["longitude"])

Chachapoyas | PE | Departamento de Amazonas | -6.23169 -77.86903
Destilería Chachapoyas | AR | Provincia de Salta | -24.74735 -65.38527
Chachapoyas | PE | Ancash | -8.97357 -78.49754
Chachapoyas | PE | Departamento de Amazonas | -6.20181 -77.85606


In [14]:
# Empezamos a limpiar los resultados.

def buscar_ciudad(nombre):
    parametros = {"name": nombre, "count": 10, "language": "es"}
    respuesta = requests.get(url_geo, params=parametros)
    datos = respuesta.json()
    return datos.get("results", [])

In [15]:
# Probamos la fórmula con la ciudad Trujillo.

buscar_ciudad("Trujillo")

[{'id': 3691175,
  'name': 'Trujillo',
  'latitude': -8.11599,
  'longitude': -79.02998,
  'elevation': 31.0,
  'feature_code': 'PPLA',
  'country_code': 'PE',
  'admin1_id': 3695781,
  'admin2_id': 3691174,
  'admin3_id': 8351227,
  'timezone': 'America/Lima',
  'population': 919899,
  'country_id': 3932488,
  'country': 'Perú',
  'admin1': 'Departamento de La Libertad',
  'admin2': 'Provincia de Trujillo',
  'admin3': 'Provincia de Trujillo'},
 {'id': 3625979,
  'name': 'Trujillo',
  'latitude': 9.36587,
  'longitude': -70.43659,
  'elevation': 795.0,
  'feature_code': 'PPLA',
  'country_code': 'VE',
  'admin1_id': 3625974,
  'admin2_id': 3625975,
  'timezone': 'America/Caracas',
  'population': 71362,
  'country_id': 3625428,
  'country': 'Venezuela',
  'admin1': 'Estado Trujillo',
  'admin2': 'Municipio Trujillo'},
 {'id': 3600591,
  'name': 'Trujillo',
  'latitude': 15.91741,
  'longitude': -85.95263,
  'elevation': 1.0,
  'feature_code': 'PPLA',
  'country_code': 'HN',
  'admin1_

In [16]:
# Aparecen varias ciudades con el mismo nombre, pero solo nos interesa la que está en La Libertad. Por eso, filtramos por país y departamento.

resultados_geo = {}

for capital in deptos["capital"]:
    resultados_geo[capital] = buscar_ciudad(capital)
    print(capital, "->", len(resultados_geo[capital]), "resultados")
    time.sleep(1)

Chachapoyas -> 4 resultados
Huaraz -> 1 resultados
Abancay -> 1 resultados
Arequipa -> 9 resultados
Ayacucho -> 10 resultados
Cajamarca -> 10 resultados
Cusco -> 5 resultados
Huancavelica -> 3 resultados
Huánuco -> 5 resultados
Ica -> 10 resultados
Huancayo -> 10 resultados
Trujillo -> 10 resultados
Chiclayo -> 4 resultados
Lima -> 10 resultados
Iquitos -> 3 resultados
Puerto Maldonado -> 2 resultados
Moquegua -> 2 resultados
Cerro de Pasco -> 2 resultados
Piura -> 8 resultados
Puno -> 10 resultados
Moyobamba -> 7 resultados
Tacna -> 10 resultados
Tumbes -> 6 resultados
Pucallpa -> 7 resultados
Callao -> 10 resultados


In [17]:
# Ahora armaremos la tabla. 

# 1. Elegimos todos los resultados que estén en Perú.

def elegir_peru(resultados):
    for r in resultados:
        if r.get("country_code") == "PE":
            return r
    return None

# Probamos la función con los resultados de Chachapoyas.

elegir_peru(resultados_geo["Chachapoyas"])

{'id': 3698540,
 'name': 'Chachapoyas',
 'latitude': -6.23169,
 'longitude': -77.86903,
 'elevation': 2360.0,
 'feature_code': 'PPLA',
 'country_code': 'PE',
 'admin1_id': 3699699,
 'admin2_id': 3698538,
 'admin3_id': 8350561,
 'timezone': 'America/Lima',
 'population': 32026,
 'country_id': 3932488,
 'country': 'Perú',
 'admin1': 'Departamento de Amazonas',
 'admin2': 'Provincia de Chachapoyas',
 'admin3': 'Chachapoyas'}

In [18]:
# 2. Armamos la tabla con las coordenadas de cada capital.

filas = []

for _, fila in deptos.iterrows():
    r = elegir_peru(resultados_geo[fila["capital"]])
    filas.append({
        "departamento": fila["departamento"],
        "capital": fila["capital"],
        "admin1": r.get("admin1") if r else None,
        "latitud": r["latitude"] if r else None,
        "longitud": r["longitude"] if r else None,
    })

geo = pd.DataFrame(filas)
geo

,departamento,capital,admin1,latitud,longitud
0,Amazonas,Chachapoyas,Departamento de Amazonas,-6.23169,-77.86903
1,Áncash,Huaraz,Ancash,-9.52614,-77.52869
2,Apurímac,Abancay,Apurímac,-13.63426,-72.88380
3,Arequipa,Arequipa,Departamento de Arequipa,-16.39899,-71.53747
4,Ayacucho,Ayacucho,Departamento de Ayacucho,-13.16380,-74.22345
5,Cajamarca,Cajamarca,Departamento de Cajamarca,-7.16378,-78.50027
6,Cusco,Cusco,Departamento de Cusco,-13.53188,-71.96701
7,Huancavelica,Huancavelica,Departamento de Huancavelica,-12.78691,-74.97298
8,Huánuco,Huánuco,Departamento de Huánuco,-9.92882,-76.23989
9,Ica,Ica,Departamento de Ica,-14.07538,-75.73422


In [19]:
# 3. Hay algunos nombres de Admin1 que no coinciden con los de la tabla de departamentos. Por ejemplo, "Lima" aparece como "Provincia de Lima". Normalizamos esos casos para poder comparar luego.

import unicodedata

def normalizar(texto):
    if texto is None:
        return ""
    texto = unicodedata.normalize("NFKD", texto)
    texto = "".join(c for c in texto if not unicodedata.combining(c))   # quita tildes
    texto = texto.lower()
    for palabra in ["departamento de ", "provincia constitucional del ", "region "]:
        texto = texto.replace(palabra, "")
    return texto.strip()

In [20]:
# Ahora, agregamos una columna con el nombre normalizado de la Admin1 y del departamento.

geo["coincide"] = [
    normalizar(d) in normalizar(a) or normalizar(a) in normalizar(d)
    for d, a in zip(geo["departamento"], geo["admin1"])
]

geo

geo[~geo["coincide"]]

,departamento,capital,admin1,latitud,longitud,coincide


In [21]:
# Imprimimos la tabla completa, otra vez.

geo

,departamento,capital,admin1,latitud,longitud,coincide
0,Amazonas,Chachapoyas,Departamento de Amazonas,-6.23169,-77.86903,True
1,Áncash,Huaraz,Ancash,-9.52614,-77.52869,True
2,Apurímac,Abancay,Apurímac,-13.63426,-72.88380,True
3,Arequipa,Arequipa,Departamento de Arequipa,-16.39899,-71.53747,True
4,Ayacucho,Ayacucho,Departamento de Ayacucho,-13.16380,-74.22345,True
5,Cajamarca,Cajamarca,Departamento de Cajamarca,-7.16378,-78.50027,True
6,Cusco,Cusco,Departamento de Cusco,-13.53188,-71.96701,True
7,Huancavelica,Huancavelica,Departamento de Huancavelica,-12.78691,-74.97298,True
8,Huánuco,Huánuco,Departamento de Huánuco,-9.92882,-76.23989,True
9,Ica,Ica,Departamento de Ica,-14.07538,-75.73422,True


In [22]:
# Definimos la temporada de lluvia por Departamento.

# Identificamos primero el periodo: del 1 de enero de 2025 al 31 de mayo de 2025

FECHA_INICIO = "2025-01-01"
FECHA_FIN = "2025-05-31"

# Establecemos la API desde donde se extraerá la información:

url_clima = "https://archive-api.open-meteo.com/v1/archive"

In [23]:
# Definimos la función 

def pedir_lluvia(lat, lon):
    parametros = {
        "latitude": lat,
        "longitude": lon,
        "start_date": FECHA_INICIO,
        "end_date": FECHA_FIN,
        "daily": "precipitation_sum",
        "timezone": "America/Lima",
    }
    respuesta = requests.get(url_clima, params=parametros)
    if respuesta.status_code != 200:
        print("  Error", respuesta.status_code)
        return None
    return respuesta.json()["daily"]["precipitation_sum"]

In [24]:
# Definimos la temporada de lluvia para las 25 capitales del Perú

lluvias = {}

for _, fila in geo.iterrows():
    print("Pidiendo:", fila["departamento"])
    lluvias[fila["departamento"]] = pedir_lluvia(fila["latitud"], fila["longitud"])
    time.sleep(1)

Pidiendo: Amazonas
Pidiendo: Áncash
Pidiendo: Apurímac
Pidiendo: Arequipa
Pidiendo: Ayacucho
Pidiendo: Cajamarca
Pidiendo: Cusco
Pidiendo: Huancavelica
Pidiendo: Huánuco
Pidiendo: Ica
Pidiendo: Junín
Pidiendo: La Libertad
Pidiendo: Lambayeque
Pidiendo: Lima
Pidiendo: Loreto
Pidiendo: Madre de Dios
Pidiendo: Moquegua
Pidiendo: Pasco
Pidiendo: Piura
Pidiendo: Puno
Pidiendo: San Martín
Pidiendo: Tacna
Pidiendo: Tumbes
Pidiendo: Ucayali
Pidiendo: Callao


In [25]:
# Definimos la lluvia diaria por temporada por cada departamento

for depto, lista in lluvias.items():
    print(depto, "->", None if lista is None else len(lista), "días")

Amazonas -> 151 días
Áncash -> 151 días
Apurímac -> 151 días
Arequipa -> 151 días
Ayacucho -> 151 días
Cajamarca -> 151 días
Cusco -> 151 días
Huancavelica -> 151 días
Huánuco -> 151 días
Ica -> 151 días
Junín -> 151 días
La Libertad -> 151 días
Lambayeque -> 151 días
Lima -> 151 días
Loreto -> 151 días
Madre de Dios -> 151 días
Moquegua -> 151 días
Pasco -> 151 días
Piura -> 151 días
Puno -> 151 días
San Martín -> 151 días
Tacna -> 151 días
Tumbes -> 151 días
Ucayali -> 151 días
Callao -> 151 días


In [26]:
# Calculamos la suma de lluvias diaria de la temporada y el número de días con 20 mm o más.

totales = []
fuertes = []

for depto in geo["departamento"]:
    validos = [x for x in lluvias[depto] if x is not None]
    totales.append(round(sum(validos), 1))
    fuertes.append(sum(1 for x in validos if x >= 20))

geo["lluvia_total_mm"] = totales
geo["dias_lluvia_fuerte"] = fuertes

geo[["departamento", "capital", "lluvia_total_mm", "dias_lluvia_fuerte"]]

,departamento,capital,lluvia_total_mm,dias_lluvia_fuerte
0,Amazonas,Chachapoyas,427.9,0
1,Áncash,Huaraz,1415.9,11
2,Apurímac,Abancay,820.0,3
3,Arequipa,Arequipa,439.4,3
4,Ayacucho,Ayacucho,500.8,1
5,Cajamarca,Cajamarca,875.7,2
6,Cusco,Cusco,801.3,4
7,Huancavelica,Huancavelica,930.4,4
8,Huánuco,Huánuco,617.0,4
9,Ica,Ica,124.3,2


In [27]:
# Nos aseguramos que los resultados estén limpios. En algunos casos puede salir "None". Para eso, calculamos cuántos días llegaron sin datos.

geo["dias_sin_dato"] = [
    sum(1 for x in lluvias[depto] if x is None)
    for depto in geo["departamento"]
]

print("Días sin dato en total:", geo["dias_sin_dato"].sum(), "de", len(geo) * 151)
geo[["departamento", "dias_sin_dato"]]

Días sin dato en total: 0 de 3775


,departamento,dias_sin_dato
0,Amazonas,0
1,Áncash,0
2,Apurímac,0
3,Arequipa,0
4,Ayacucho,0
5,Cajamarca,0
6,Cusco,0
7,Huancavelica,0
8,Huánuco,0
9,Ica,0


In [33]:
# Ahora importamos la tabla como archivo .CSV.
# Consideramos que incluya las columnas: departamento, capital, latitud, longitud, lluvia_total_mm y dias_lluvia_fuerte.

import os

columnas = ["departamento", "capital", "latitud", "longitud", "lluvia_total_mm", "dias_lluvia_fuerte"]
final = geo[columnas].copy()

os.makedirs("datos", exist_ok=True)
final.to_csv("datos/lluvias_por_departamento.csv", index=False, encoding="utf-8")